## OPTIMAL EVENT TRADING

Optimal execution around scheduled events (e.g. earnings, macro announcements): modeling optimal timing and sizing of trades around a known event time.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard library imports

# Third-party imports
import numpy as np
import polars as pl
from bokeh.io import output_notebook

# First-party imports
from xpectral.quant.execution import (
    efficient_frontier,
    optimal_holdings,
    permanent_impact_cost,
    temporary_impact_cost,
)

output_notebook(hide_banner=True)

## Optimal holdings trajectory

The Almgren-Chriss closed-form schedule, $x(t) = X \cdot \sinh(\kappa(T-t)) / \sinh(\kappa T)$,
for the three illustrative cases from `almgren-chriss-explained.md`: risk-neutral
($\lambda=0$, a straight line), moderate ($\kappa=2$), and aggressive ($\kappa=5$).

In [3]:
X = 1.0
T = 1.0
t_grid = np.linspace(0.0, T, 101)

kappas = {"risk_neutral": 0.0, "moderate": 2.0, "aggressive": 5.0}
colors = {"risk_neutral": "#1f77b4", "moderate": "#ff7f0e", "aggressive": "#2ca02c"}

trajectories_df = pl.DataFrame(
    {
        "t_fraction": t_grid / T,
        **{
            label: optimal_holdings(t_grid, X, T, kappa)
            for label, kappa in kappas.items()
        },
    }
)

fig = trajectories_df.bokeh(
    title="Optimal holdings trajectory x(t)/X",
    x_axis_label="Fraction of time elapsed",
    y_axis_label="Fraction of shares held",
    width=700,
    height=400,
)
for label, kappa in kappas.items():
    legend_suffix = "λ=0" if kappa == 0.0 else f"κ={kappa:g}"
    fig.line(
        x="t_fraction",
        y=label,
        legend_label=f"holdings ({legend_suffix})",
        line_width=2,
        color=colors[label],
    )
fig.legend.location = "center_right"
fig.show()

## Efficient frontier

Sweeping the risk-aversion parameter $\lambda$ traces out the efficient frontier:
the lowest achievable expected cost for each level of cost variance (risk). Expected
cost splits into a permanent-impact piece, $\frac{1}{2}\gamma X^2$, fixed by the total
size $X$ and identical for every schedule, and a temporary-impact piece that shrinks as
the schedule spreads trading over more time (lower risk aversion, lower $\kappa$).

In [4]:
# Illustrative liquidation of 1,000,000 shares over one trading day.
X = 1_000_000.0
T = 1.0
sigma = 0.5  # $/share/sqrt(day)
eta = 2.5e-6  # temporary impact coefficient
gamma = 2.5e-7  # permanent impact coefficient

lambdas = np.concatenate([[0.0], np.logspace(-10, -2, 49)])

frontier = efficient_frontier(lambdas, X, T, sigma, eta, gamma)
frontier_df = pl.DataFrame(
    {
        "variance": frontier["variance"],
        "expected_cost": frontier["expected_cost"],
        "permanent_cost": permanent_impact_cost(X, gamma),
        "temporary_cost": [
            temporary_impact_cost(X, eta, T, kappa) for kappa in frontier["kappa"]
        ],
    }
).sort("variance")

fig = frontier_df.bokeh(
    title="Almgren-Chriss efficient frontier and cost composition",
    x_axis_label="Variance of cost ($²)",
    y_axis_label="Cost ($)",
    width=700,
    height=400,
)
fig.line(x="variance", y="expected_cost", legend_label="expected cost", line_width=2)
fig.scatter(x="variance", y="expected_cost", legend=False, size=5)
fig.line(
    x="variance",
    y="permanent_cost",
    legend_label="permanent-impact cost",
    line_width=2,
    line_dash="dashed",
)
fig.line(
    x="variance",
    y="temporary_cost",
    legend_label="temporary-impact cost",
    line_width=2,
    line_dash="dotted",
)
fig.legend.location = "top_right"
fig.show()